In [2]:
from langchain_community.document_loaders import PyPDFLoader

# 문서 로드
loader = PyPDFLoader('../data/KCI_FI003153549.pdf')
documents = loader.load()

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.split_documents(documents)

In [8]:
len(chunks)

42

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

# 임베딩 모델
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=gemini_api_key,
)

#### ⚠️ Gemini API의 할당량(Quota) 제한을 초과 발생

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
import uuid

# 문서 요약
prompt_text = '다음 문서의 요약을 생성하세요:\n\n{doc}'

prompt = ChatPromptTemplate.from_template(prompt_text)

# 문서 요약에 사용할 모델
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0, google_api_key=gemini_api_key)
    
summarize_chain = {
    'doc': lambda x: x.page_content} | prompt | llm | StrOutputParser()

summaries = summarize_chain.batch(chunks, {'max_concurrency': 5})

id_key = 'doc_id'

# 문서와 동일한 길이가 필요하므로 summaries에서 chunks로 변경
doc_ids = [str(uuid.uuid4()) for _ in chunks]

# 각 요약은 doc_id를 통해 원본 문서와 연결
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 50
}
].
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing

#### ✅ 지연 시간 추가하여 회피

In [6]:
# ! 요약시  Gemini Free Tier 분당 10회 제한 초과로 인한 429 발생
# time 모듈을 사용하여 각 배치 호출 사이에 지연 시간(delay)을 추가하는 방법으로 회피
import time

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
import uuid

# 문서 요약
prompt_text = '다음 문서의 요약을 생성하세요:\n\n{doc}'

prompt = ChatPromptTemplate.from_template(prompt_text)

# 문서 요약에 사용할 모델
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0, google_api_key=gemini_api_key)
    
summarize_chain = {
    'doc': lambda x: x.page_content} | prompt | llm | StrOutputParser()

# --- 수정된 부분 ---
summaries = []
batch_size = 5  # 한 번에 처리할 문서 조각 수

# chunks 리스트를 batch_size 단위로 순회
for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    # 각 배치에 대해 요약 작업 수행
    # batch() 메서드는 여러 입력(문서 조각)을 한 번에 처리(병렬 처리)
    batch_summaries = summarize_chain.batch(batch_chunks, {'max_concurrency': batch_size})
    summaries.extend(batch_summaries)
    
    # 마지막 배치가 아닌 경우에만 대기
    if i + batch_size < len(chunks):
        print(f"Processed {i + batch_size} chunks. Waiting for 60 seconds to avoid API limit...")
        time.sleep(60)

# -----------------

id_key = 'doc_id'

doc_ids = [str(uuid.uuid4()) for _ in chunks]

summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

Processed 5 chunks. Waiting for 60 seconds to avoid API limit...
Processed 10 chunks. Waiting for 60 seconds to avoid API limit...
Processed 15 chunks. Waiting for 60 seconds to avoid API limit...
Processed 20 chunks. Waiting for 60 seconds to avoid API limit...
Processed 25 chunks. Waiting for 60 seconds to avoid API limit...
Processed 30 chunks. Waiting for 60 seconds to avoid API limit...
Processed 35 chunks. Waiting for 60 seconds to avoid API limit...
Processed 40 chunks. Waiting for 60 seconds to avoid API limit...


In [10]:
summary_docs[0]

Document(metadata={'doc_id': 'f30db245-3721-40a3-bc62-55bd3b4d933d'}, page_content="이 문서는 한국컴퓨터정보학회논문지(JKSCI) 2024년 12월호(Vol. 29 No. 12, pp. 169-180)에 게재된 'Clinical Trials Utilizing LLM-Based Generative AI'라는 제목의 논문입니다.\n\n이 연구는 의료기기 임상시험 분야에 LLM(대규모 언어 모델) 기반의 Private LLM을 적용하여 업무 효율성과 전문성을 향상시키는 방안을 탐구하는 것을 목적으로 합니다.")

In [ ]:
from langchain_community.vectorstores import FAISS

# 벡터 저장소(FAISS)에는 요약을 인덱싱하고, 원문은 별도 docstore 폴더로 관리
vectorstore = FAISS.from_documents(summary_docs, embeddings_model)

In [21]:
# FAISS 벡터 저장소를 로컬 파일에 저장
vectorstore.save_local("./faiss_index")

In [11]:
from langchain.storage import LocalFileStore, create_kv_docstore

# 파일 기반 문서 저장소 구성
byte_store = LocalFileStore("./docstore")
store = create_kv_docstore(byte_store)

In [ ]:
# 원본 문서를 문서 저장소에 저장 (doc_id로 연결)
# 문서 내용은 유니코드 형식(\u)로 저장
store.mset(list(zip(doc_ids, chunks)))

In [15]:
from langchain.retrievers.multi_vector import MultiVectorRetriever

# MultiVectorRetriever 구성
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)


In [16]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

#### 요약 검색

In [ ]:
# 벡터 저장소가 요약을 검색
sub_docs = retriever.vectorstore.similarity_search(query, k=2)

In [26]:
sub_docs

[Document(id='d859f6fe-9454-4302-bb9d-7e14507dff92', metadata={'doc_id': '72e07c18-b086-47e1-b956-45d4cbb3f029'}, page_content='이 문서는 확립된 시스템을 활용하여 전문가 수준의 Q&A 기능을 제공하고 임상시험 운영 중 발생하는 복잡한 문제들을 해결하는 Private LLM 시스템에 대해 설명합니다. 연구 결과, 이 시스템은 작업 자동화 및 정밀한 의사결정 지원에서 기존 방법론을 능가했습니다. 특히, 도메인 전문가의 질문에 정확한 답변을 제공하고 새로운 임상 기준 및 통찰력을 생성하는 능력은 의료기기 임상시험 운영에서 혁신적인 도구가 될 잠재력을 보여주었습니다. 이는 정밀 의료, 임상시험 관리 자동화, 그리고 도메인 지식 기반 Q&A 시스템에서 Private LLM의 실용적 적용 가능성을 확인시켜 주었습니다.'),
 Document(id='b68b5b8c-3c4a-40bb-98f8-0d4265686034', metadata={'doc_id': '8e4cceb9-fc06-4b5f-8be8-297e6b2fe3ed'}, page_content='의료기기 임상시험 분야의 Private LLM 튜닝을 위해 총 158개의 문서(11,954 페이지)가 수집되었다. 이 문서들은 규제 문서(30%), 교육 자료(20%), 프로토콜 및 보고서(25%), 의료기기 특화 문서(15%), 기타(10%)로 분류된다. 수집된 데이터셋은 의료기기 임상시험의 규제, 프로토콜 설계, 데이터 관리 등 전반적인 지식을 포괄하며, 도메인 적합성, 다양성 및 응용 가능성 측면에서 높은 타당성을 갖추고 있다.')]

In [27]:
print(sub_docs[0].page_content)

이 문서는 확립된 시스템을 활용하여 전문가 수준의 Q&A 기능을 제공하고 임상시험 운영 중 발생하는 복잡한 문제들을 해결하는 Private LLM 시스템에 대해 설명합니다. 연구 결과, 이 시스템은 작업 자동화 및 정밀한 의사결정 지원에서 기존 방법론을 능가했습니다. 특히, 도메인 전문가의 질문에 정확한 답변을 제공하고 새로운 임상 기준 및 통찰력을 생성하는 능력은 의료기기 임상시험 운영에서 혁신적인 도구가 될 잠재력을 보여주었습니다. 이는 정밀 의료, 임상시험 관리 자동화, 그리고 도메인 지식 기반 Q&A 시스템에서 Private LLM의 실용적 적용 가능성을 확인시켜 주었습니다.


#### 요약 검색 후 원본 문서 검색

**검색 과정**

1. **벡터 저장소(FAISS)**에 질의를 던져 유사한 요약 문서 찾음

2. 찾아낸 요약 문서의 메타데이터에 있는 doc_id 추출

3. 추출된 doc_id를 사용하여 **문서 저장소(docstore)**에서 원본 문서 검색

4. 최종적으로 원본 문서 청크 반환

In [19]:
# retriever는 더 큰 원본 문서 청크를 반환
retrieved_docs = retriever.invoke(query)

In [25]:
len(retrieved_docs)

4

In [22]:
retrieved_docs

[Document(metadata={'producer': 'ezPDF Builder Supreme', 'creator': 'PyPDF', 'creationdate': '2024-12-27T02:09:00+09:00', 'moddate': '2024-12-27T02:09:00+09:00', 'source': '../data/KCI_FI003153549.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='realizes expert-level Q&A function by utilizing the established system and solves complex questions and \nproblems that arise during clinical trial operation. Finally, by evaluating the performance of the system, \nwe propose a direction to increase the efficiency and reliability of clinical trial operation and medical \ndevice development. As a result of the study, the Private LLM system has outperformed the existing \nmethodology in supporting task automation and precise decision making. In particular, the ability to \nprovide accurate answers to questions from domain experts and to generate new clinical criteria and \ninsights shows the potential to become an innovative tool in clinical trial operation of medical devices

In [24]:
retrieved_docs[0].page_content

'realizes expert-level Q&A function by utilizing the established system and solves complex questions and \nproblems that arise during clinical trial operation. Finally, by evaluating the performance of the system, \nwe propose a direction to increase the efficiency and reliability of clinical trial operation and medical \ndevice development. As a result of the study, the Private LLM system has outperformed the existing \nmethodology in supporting task automation and precise decision making. In particular, the ability to \nprovide accurate answers to questions from domain experts and to generate new clinical criteria and \ninsights shows the potential to become an innovative tool in clinical trial operation of medical devices. \nThis confirmed the practical applicability of Private LLM in precision medical care, automation of \nclinical trial management, and a Q&A system based on domain knowledge.'